<a href="https://colab.research.google.com/github/medwicks/BAN482-MachineLearning/blob/main/BAN482_DecisionTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Decision Trees - Regression

#### Import Libraries

In [1]:
# ==============================================
# IMPORT PACKAGES AND FUNCTIONS
# ==============================================

import pandas as pd
# We'll use this to load our penguin dataset and work with it

import seaborn as sns
# seaborn (nicknamed 'sns'): A library for creating beautiful statistical visualizations
# - Makes it easy to create professional-looking charts and graphs
# - Built on top of matplotlib but easier to use for statistical plots

import matplotlib.pyplot as plt
# matplotlib.pyplot (nicknamed 'plt'): A foundational plotting library
# - Used for creating and customizing all types of charts and figures
# - We use this to control figure sizes and display our visualizations

from sklearn.tree import DecisionTreeRegressor, export_text
# DecisionTreeRegressor: The main machine learning model we'll use
# - 'Regressor' means it predicts numerical values (in our case, penguin body mass)
# - It builds a tree of if-then rules based on the data to make predictions
# export_text: A helper function to display the tree rules as readable text
# - Converts the decision tree into text format so we can see exactly what rules it learned

from sklearn import tree, metrics
# tree: Contains additional functions for visualizing and working with decision trees
# - Includes functions like plot_tree() which creates visual diagrams of our tree
# metrics: Functions for measuring how accurate our predictions are
# - Contains functions like mean_squared_error() to calculate how far off our predictions are

# import matplotlib_inline.backend_inline
# matplotlib_inline.backend_inline: Ensures plots display properly inside this Jupyter notebook
# - Without this, our plots might not show up correctly when we run the code
# - This is specific to Jupyter/Colab notebooks

#### Data Loading and Pre-Processing

In [7]:
# ==============================================
# LOAD AND PREPARE THE DATA
# ==============================================

# Step 1: Load the penguins dataset from a CSV file
# pd.read_csv() reads data from a file and creates a DataFrame (a table of data)
penguins = pd.read_csv('palmer_penguins/MyDrive/SeniorFall/BAN482.csv')

# Step 2: Remove rows with missing data
# .dropna() removes any rows that have missing values (NaN = "Not a Number")
# Missing data can cause errors in our analysis, so we remove those rows
# This creates a cleaner dataset with only complete observations
penguins = penguins.dropna()

# Step 3: Filter to only Gentoo penguins
# We're focusing our analysis on just one species: Gentoo penguins
# penguins['species'] == 'Gentoo' creates a True/False (Boolean) mask for Gentoo rows
# penguins[...] selects only the rows where the mask is True
# .copy() creates a separate copy so changes don't affect the original data
gentoo = penguins[penguins['species'] == 'Gentoo'].copy()

# Step 4: Display summary statistics for the Gentoo penguin data
# .describe() calculates statistical summaries: count, mean, std, min, max, etc.
# include='all' tells it to include both numerical and categorical columns
# This helps us understand the distribution and range of our data
gentoo.describe(include='all')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
count,119,119,119.000000,119.000000,119.000000,119.000000,119,119.000000
unique,1,1,NaN,NaN,NaN,NaN,2,NaN
top,Gentoo,Biscoe,NaN,NaN,NaN,NaN,male,NaN
freq,119,119,NaN,NaN,NaN,NaN,61,NaN
mean,NaN,NaN,47.568067,14.996639,217.235294,5092.436975,NaN,2008.067227
std,NaN,NaN,3.106116,0.985998,6.585431,501.476154,NaN,0.789025
min,NaN,NaN,40.900000,13.100000,203.000000,3950.000000,NaN,2007.000000
25%,NaN,NaN,45.350000,14.200000,212.000000,4700.000000,NaN,2007.000000
50%,NaN,NaN,47.400000,15.000000,216.000000,5050.000000,NaN,2008.000000
75%,NaN,NaN,49.600000,15.750000,221.500000,5500.000000,NaN,2009.000000


### Setting up the Scikit Learn Workflow

#### DEFINE

In [8]:
# ==============================================
# CREATE THE FEATURE MATRIX (X)
# ==============================================
# Select the columns (features) we'll use to predict penguin body mass
# In machine learning, we call the input variables "features" and typically name them "X"
# X (uppercase) is a standard name for the feature matrix
# This is a 2D table where:
#   - Each row is one penguin
#   - Each column is one measurement (sex, flipper length, bill length)
# We're selecting these specific features because they're likely related to body mass:
#   - 'sex': Male penguins are typically larger
#   - 'flipper_length_mm': Bigger flippers might indicate a larger penguin
#   - 'bill_length_mm': Longer bills might indicate a larger penguin
# The double brackets [[ ]] create a DataFrame (table) rather than a single column
X = gentoo[['sex', 'flipper_length_mm', 'bill_length_mm']]

# Display the feature matrix to see what we're working with
# This shows us the data that will be used to train our decision tree model
X

,sex,flipper_length_mm,bill_length_mm
152,female,211.0,46.1
153,male,230.0,50.0
154,female,210.0,48.7
155,male,218.0,50.0
156,male,215.0,47.6
...,...,...,...
270,female,214.0,47.2
272,female,215.0,46.8
273,male,222.0,50.4
274,female,212.0,45.2


`DecisionTreeRegressor` only takes numerical values as features, which means features like `sex` and `island` won't work.

Thus, categorical features must be encoded as dummy variables that indicate which category the individual falls into.  This encoding is done in `pandas` with `get_dummies`.

In [ ]:
# ==============================================
# CONVERT CATEGORICAL VARIABLES TO DUMMY VARIABLES
# ==============================================
# Decision trees (and most machine learning models) can only work with numbers
# But our 'sex' column contains text: 'male' or 'female'
# We need to convert this text into numbers using "dummy variables" (also called "one-hot encoding")

# What are dummy variables?
# - They're binary (0 or 1) variables that represent categories
# - For sex: we create a new column called 'sex_male'
#   - sex_male=1 means the penguin is male
#   - sex_male=0 means the penguin is female

# pd.get_dummies() automatically creates these dummy variables
# Parameters:
#   - X: The feature matrix we created earlier (contains sex, flipper_length_mm, bill_length_mm)
#   - drop_first=True: Creates only ONE dummy column instead of two
#     - Without this, we'd get 'sex_male' AND 'sex_female' columns
#     - But we only need ONE because: if sex_male=0, we know it's female
#     - This prevents "dummy variable trap" (redundant information that confuses models)
# The result is a new DataFrame where:
#   - 'sex' is replaced with 'sex_male' (0 or 1)
#   - The numerical columns (flipper_length_mm, bill_length_mm) stay the same
X_dummies = pd.get_dummies(X, drop_first=True)

# Display the converted feature matrix
# You should see 'sex_male' column with 0s and 1s instead of 'male'/'female'
X_dummies

The `drop_first=True` option creates a single dummy variable, instead of two. One dummy variable completely describes sex in this dataset. 0 and 1 are assigned to classes in alphabetical order.

- `sex_male=0`: female
- `sex_male=1`: male

In [9]:
# ==============================================
# CREATE THE TARGET VARIABLE (Y)
# ==============================================
# Extract the target variable - what we want to predict
# y is a common name for the target/dependent variable in machine learning
# We're trying to predict 'body_mass_g' (penguin body mass in grams)
y = gentoo['body_mass_g']

#### INITIALIZE

In [ ]:


# ==============================================
# CREATE AND CONFIGURE THE DECISION TREE MODEL
# ==============================================
# Create a decision tree regressor model with specific parameters
# DecisionTreeRegressor: A machine learning model that makes predictions by learning decision rules
# max_depth=2: Limits the tree to 2 levels of decisions (prevents overfitting)
#   - This means the tree will ask at most 2 questions before making a prediction
#   - Deeper trees can memorize training data but may not predict new data well
# min_samples_leaf=2: Each final leaf (prediction) must have at least 2 samples
#   - This prevents the tree from making predictions based on single unusual observations
#   - Helps make the model more generalizable and less sensitive to outliers
regtreeModel = DecisionTreeRegressor(max_depth=2, min_samples_leaf=2);



#### FIT

In [ ]:
# ==============================================
# TRAIN (FIT) THE MODEL
# ==============================================
# Train the decision tree model on our data
# .fit() is the method that makes the model "learn" from the data
# X_dummies: The input features (flipper length, bill length, sex)

# y: The target we want to predict (body mass)
# After this step, the model has learned rules like:
#   "If flipper length > 210mm AND sex is male, predict 5551g"
regtreeModel.fit(X_dummies, y);

In [ ]:
# ==============================================
# DISPLAY THE DECISION TREE AS TEXT
# ==============================================
# After training our model, we want to see what rules it learned
# The decision tree creates if-then rules, and we can view them as text

# export_text() converts the decision tree model into a readable text format
# Parameters:
#   - regtreeModel: The trained decision tree model we created earlier
#   - feature_names=X.columns.to_list(): Tells it to use our actual feature names
#     (sex, flipper_length_mm, bill_length_mm) instead of generic names like "X[0]"
# This creates a text representation showing:
#   - The decision rules at each node (e.g., "if flipper_length_mm <= 210.5")
#   - The predicted body mass values at each leaf (final prediction)
#   - The structure of how the tree makes decisions

# print() displays the tree rules to the screen
# You'll see output like:
#   |--- flipper_length_mm <= 210.5
#   |   |--- value: [4806.25]  (predicted body mass for small flippers)
#   |--- flipper_length_mm >  210.5
#   |   |--- sex_male <= 0.5
#   |   |   |--- value: [4472.73]  (predicted body mass for female with large flippers)
#   |   |--- sex_male >  0.5
#   |   |   |--- value: [5551.56]  (predicted body mass for male with large flippers)
print(export_text(regtreeModel, feature_names=X.columns.to_list()))

In [ ]:
# ==============================================
# VISUALIZE THE DECISION TREE AS A DIAGRAM
# ==============================================
# Now we'll create a visual diagram of our decision tree
# This shows the tree structure with boxes and branches

# Step 1: Create a figure (plotting window) with a specific size
# plt.figure() creates a new blank canvas for our plot
# figsize=[12, 8] sets the size: 12 inches wide by 8 inches tall
#   - Larger figures make the tree easier to read
#   - Without this, the default size might be too small to see details
plt.figure(figsize=[12, 8])

# Step 2: Use tree.plot_tree() to draw the decision tree diagram
# This function creates a visual representation of the tree structure
p = tree.plot_tree(
    regtreeModel,
    feature_names=X.columns,
    class_names=y.unique(),
    filled=False,
    fontsize=12
)
# Parameters:
#   - regtreeModel: The trained decision tree model we want to visualize
#   - feature_names=X.columns: Labels each node with actual feature names
#     (sex, flipper_length_mm, bill_length_mm) instead of generic "X[0]"
#   - class_names=y.unique(): Shows the unique body mass values at leaf nodes
#     (Though for regression trees, this shows the numerical predictions)
#   - filled=False: Don't color the boxes (when True, colors indicate value/class)
# The diagram will show:
#   - Rectangular boxes representing each decision node
#   - The splitting criterion (e.g., "flipper_length_mm <= 210.5")
#   - The number of samples at each node
#   - The predicted value (mean body mass) at each leaf node
#   - Lines connecting parent nodes to child nodes
# This visual helps us understand exactly how the tree makes decisions


#### PREDICT

In [ ]:
# ==============================================
# MAKE PREDICTIONS AND ADD THEM TO THE DATASET
# ==============================================
# Now that our model is trained, we can use it to make predictions
# We'll predict the body mass for each penguin and add those predictions to our dataset

# Step 1: Use the trained model to make predictions
# regtreeModel.predict(X_dummies) applies the decision tree rules to make predictions
#   - For each penguin, it follows the tree's if-then rules
#   - Example: "If flipper_length > 210.5 AND sex_male = 1, predict 5551g"
#   - Returns an array of predicted body mass values (one for each penguin)

# Step 2: Add the predictions as a new column called 'pred' in the gentoo dataset
# gentoo['pred'] = ... creates a new column named 'pred' and stores the predictions there
# This allows us to compare:
#   - gentoo['body_mass_g']: The actual, observed body mass
#   - gentoo['pred']: The predicted body mass from our model
# By adding predictions to the original dataset, we can easily:
#   - Calculate how accurate our predictions are
#   - Visualize the relationship between predicted and observed values
#   - Identify which penguins our model predicted well vs. poorly
gentoo['pred'] = regtreeModel.predict(X_dummies)

# Display the dataset with the new 'pred' column
# You'll see the original columns plus the new predictions column
gentoo

#### EVALUATE

In [ ]:
# ==============================================
# VISUALIZE PREDICTED VS OBSERVED VALUES
# ==============================================
# Now we'll create a scatter plot to see how well our predictions match reality
# This visual comparison helps us understand the model's accuracy

# Create a scatter plot using seaborn
# sns.scatterplot() creates a scatter plot (dots representing individual penguins)
p = sns.scatterplot(data=gentoo, x='body_mass_g', y='pred', hue='sex')
# Parameters:
#   - data=gentoo: Use the gentoo dataset (which now includes our predictions)
#   - x='body_mass_g': Put the ACTUAL observed body mass on the x-axis (horizontal)
#   - y='pred': Put the PREDICTED body mass on the y-axis (vertical)
#   - hue='sex': Color the dots by sex (males and females will be different colors)
# If our model is perfect, all dots would fall on a diagonal line
# (because predicted would equal observed)
# Dots above the line: Model overpredicted (predicted too high)
# Dots below the line: Model underpredicted (predicted too low)


# Customize the x-axis label
p.set_xlabel('Observed body mass', fontsize=14)
# p.set_xlabel() sets the label for the horizontal axis
# 'Observed body mass': Clear, descriptive label
# fontsize=14: Makes the label text larger and easier to read


# Customize the y-axis label
p.set_ylabel('Predicted body mass', fontsize=14)
# p.set_ylabel() sets the label for the vertical axis
# 'Predicted body mass': Clear, descriptive label
# fontsize=14: Makes the label text larger and easier to read


In [ ]:
# ==============================================
# CALCULATE THE MEAN SQUARED ERROR (MSE)
# ==============================================
# The final step is to measure how accurate our predictions are
# We'll use Mean Squared Error (MSE), a common metric for regression models

# What is MSE?
# - MSE measures the average squared difference between predicted and actual values
# - Lower MSE = better predictions (closer to actual values)
# - Formula: MSE = average of (predicted - actual)²
# - Squaring the errors:
#   1. Makes all errors positive (no canceling out)
#   2. Penalizes large errors more heavily than small errors

# metrics.mean_squared_error() calculates the MSE
# Parameters:
#   - gentoo['pred']: Our predicted body mass values from the decision tree
#   - y: The actual observed body mass values (what we're trying to predict)
# The function:
#   1. Calculates the difference for each penguin: (predicted - actual)
#   2. Squares each difference: (predicted - actual)²
#   3. Takes the average of all squared differences
# Example interpretation:
#   - If MSE = 40,000, the average squared error is 40,000 grams²
#   - Taking the square root: √40,000 ≈ 200 grams
#   - This means our predictions are off by about 200 grams on average
# A lower MSE indicates our decision tree model is making more accurate predictions
metrics.mean_squared_error(gentoo['pred'], y)

## Decision Trees - Classification

#### Import Libraries

In [ ]:
# Import packages and functions
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn import metrics, tree

#### Loading Data and Pre-Processing

In [ ]:
# Load the penguins data
penguins = pd.read_csv('palmer_penguins.csv')

# Drop penguins with missing values
penguins = penguins.dropna()

# Calculate summary statistics using .describe()
penguins.describe(include='all')

### Setting up the Scikit Learn Workflow

#### DEFINE

In [ ]:
# Save output features as y
y = penguins[['species']]

# Save input features as x
X = penguins[['flipper_length_mm', 'bill_length_mm']]

#### INITIALIZE

In [ ]:
# Initialize the model
classtreeModel = DecisionTreeClassifier(max_depth=2)

#### FIT

In [ ]:
# Fit the model
classtreeModel = classtreeModel.fit(X, y)

In [ ]:
# Print tree as text
print(export_text(classtreeModel, feature_names=X.columns.to_list()))

In [ ]:
# Resize the plotting window
plt.figure(figsize=[12, 8])

# Values in brackets represent classes in alphabetical order
# [Adelie, Chinstrap, Gentoo]
p = tree.plot_tree(classtreeModel, feature_names=X.columns, filled=False, fontsize=10)

#### PREDICT

In [ ]:
y_pred = classtreeModel.predict(X)

#### EVALUATE

In [ ]:
# Calculate the confusion matrix
metrics.confusion_matrix(y, y_pred)

# Plot the confusion matrix
metrics.ConfusionMatrixDisplay.from_predictions(y, y_pred)

In [ ]:
# Calculate the Gini index
probs = pd.DataFrame(data=classtreeModel.predict_proba(X))

print("Gini index: ", (probs * (1 - probs)).mean().sum())

## Random Forest: Regression

### Import libraries

In [ ]:
# Import the pandas library for data manipulation and analysis (working with dataframes)
import pandas as pd
# Import matplotlib's pyplot module for creating basic plots and visualizations
import matplotlib.pyplot as plt
# Import seaborn library for enhanced and more attractive statistical visualizations
import seaborn as sns
# Import metrics and tree modules from scikit-learn: metrics for evaluating models, tree for visualizing decision trees
from sklearn import metrics, tree
# Import RandomForestRegressor class which implements the random forest algorithm for regression tasks
from sklearn.ensemble import RandomForestRegressor
# Import train_test_split function to divide our dataset into separate training and testing portions
from sklearn.model_selection import train_test_split

### Loading Data and Preprocessing

In [ ]:
# Load the Palmer Penguins dataset from a CSV file into a pandas DataFrame
penguins = pd.read_csv('palmer_penguins.csv')
# Display summary statistics for all columns including count, mean, std, min, max, and quartiles
penguins.describe(include='all')

#### Data Cleaning: Removing rows with missing values

In [ ]:
# Remove all rows that contain any missing (NaN) values to ensure clean data for modeling
penguins = penguins.dropna()

#### Encoding categorical variables

In [ ]:
# Random forest models require all numerical inputs (they cannot process categorical text data)
# Convert categorical features like species and island into binary indicator variables (one-hot encoding)
penguinDummies = pd.get_dummies(penguins, drop_first=True)
# Example: species_Chinstrap column will contain 1 for Chinstrap penguins, 0 for all others
# The drop_first=True parameter prevents data redundancies by dropping one category level
penguinDummies

### Setting up the Scikit Learn Workflow

#### DEFINE

In [ ]:
# Extract the target variable 'body_mass_g' (what we want to predict) into variable y
y = penguinDummies["body_mass_g"]
# Create the feature matrix X by removing the target variable from the dataset
# axis=1 means we're dropping a column (not a row)
X = penguinDummies.drop("body_mass_g", axis=1)
# Split the data into training and testing sets for model evaluation
# test_size=0.3 means 30% of data is reserved for testing, 70% for training
# random_state=8675309 ensures we get the same split every time we run the code (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=8675309)


#### INITIALIZE

In [ ]:
# Initialize a RandomForestRegressor model with specific hyperparameters
# max_depth=2 limits each tree to 2 levels deep (prevents overfitting)
# max_features='sqrt' means each tree considers sqrt(n_features) random features at each split
# random_state=99 ensures reproducible results
rfModel = RandomForestRegressor(max_depth=2, max_features='sqrt', random_state=99)

#### FIT

In [ ]:
# Train the random forest model using the training data
# The model learns patterns from X_train to predict y_train
rfModel.fit(X_train, y_train);

Inspect Variable Importance

In [ ]:
# Create a DataFrame to display feature importance scores from the trained model
# feature_names_in_ contains the names of all input features used during training
# feature_importances_ contains importance scores (higher = more important for predictions)
pd.DataFrame(
    data={
        'feature': rfModel.feature_names_in_,
        'importance': rfModel.feature_importances_,
    }
# Sort the features by importance in descending order (most important first)
).sort_values('importance', ascending=False)

#### PREDICT

In [ ]:
# Use the trained random forest model to make predictions on the test set
# X_test contains the features for penguins the model has never seen during training
y_pred = rfModel.predict(X_test)

#### EVALUATE

Compute the test MSE

In [ ]:
# Calculate and display the Mean Squared Error (MSE) to evaluate model performance
# MSE measures average squared difference between actual and predicted values
# Lower MSE indicates better model performance (closer predictions to actual values)
mse = metrics.mean_squared_error(y_test, y_pred)
print("MSE: ", mse)

Compare Predictions with Actual values

In [ ]:
# Create a scatterplot comparing actual test values (x-axis) to predicted values (y-axis)
# This helps visualize how well the model predictions match the true body mass values
p = sns.scatterplot(x=y_test, y=y_pred)
# Set the x-axis label to indicate these are the actual measured body mass values
p.set_xlabel("Actual values", fontsize=14)
# Set the y-axis label to indicate these are the model's predicted body mass values
p.set_ylabel("Predicted values", fontsize=14)
# Add a red dashed diagonal reference line from point (3000, 3000) to (6000, 6000)
# If predictions were perfect, all points would fall exactly on this line
# Points above the line = overestimated, points below = underestimated
plt.axline((3000, 3000), (6000, 6000), color='r', ls='--')

Accessing individual trees that are part of the random forest

In [ ]:
# Extract the first individual decision tree from the random forest ensemble
# estimators_ is a list containing all trees in the forest; [0] selects the first one
singleTree = rfModel.estimators_[0]
# Create a figure with specified dimensions: 18 inches wide by 6 inches tall
# This ensures the tree visualization is large enough to read clearly
plt.figure(figsize=[18, 6])
# Visualize the structure of the single decision tree
# feature_names=X.columns labels nodes with actual feature names instead of generic labels
# filled=False keeps nodes uncolored (set to True for colored nodes based on value)
# fontsize=10 sets readable text size for node labels
tree.plot_tree(singleTree, feature_names=X.columns, filled=False, fontsize=10)

## Random Forest: Classification

### Import Libraries

In [ ]:
# Import pandas for data manipulation
import pandas as pd
# Import matplotlib for visualizations
import matplotlib.pyplot as plt
# Import metrics and tree from sklearn
from sklearn import metrics, tree
# Import RandomForestClassifier for classification
from sklearn.ensemble import RandomForestClassifier
# Import train_test_split to split data
from sklearn.model_selection import train_test_split

### Loading data and preprocessing

In [ ]:
# Load the penguin dataset from a CSV file into a pandas DataFrame
penguins = pd.read_csv('palmer_penguins.csv')
# Display summary statistics for all columns (both numeric and categorical)
penguins.describe(include='all')

#### Data Cleaning: Removing rows with missing values

In [ ]:
# Remove any rows from the dataset that have missing values
penguins = penguins.dropna()

#### DEFINE

In [ ]:
# Set y as the target variable (species column) that we want to predict
y = penguins['species']
# Set X as all the input features by removing the species column from the dataset
X = penguins.drop('species', axis=1)
# Convert categorical variables like island and sex into binary columns (0 or 1)
X = pd.get_dummies(X, drop_first=True)
# Display the transformed input features
X

In [ ]:
# Split the data into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

#### INITIALIZE

In [ ]:
# Create a random forest classifier with max tree depth of 2 and square root feature selection
rfModel = RandomForestClassifier(max_depth=2, max_features='sqrt', random_state=99);

#### FIT

In [ ]:
# Train the random forest model using the training data
rfModel.fit(X_train, y_train);

Inspecting Feature Importance

In [ ]:
# Create a DataFrame showing each feature's importance in the model
pd.DataFrame(
    data={
        'feature': rfModel.feature_names_in_,
        'importance': rfModel.feature_importances_,
    }
# Sort features by importance from highest to lowest
).sort_values('importance', ascending=False)

#### PREDICT

In [ ]:
# Use the trained model to predict species for the test data
y_pred = rfModel.predict(X_test)

Creating Transfer Matrix to evaluate test error

In [ ]:
# Generate a confusion matrix comparing actual vs predicted species
metrics.confusion_matrix(y_test, y_pred)
# Create a visual display of the confusion matrix
metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

In [ ]:
# Get the predicted probabilities for each species class from the model
probs = pd.DataFrame(data=rfModel.predict_proba(X_test))
# Calculate the Gini index which measures classification purity
print("Gini index ", (probs * (1 - probs)).mean().sum())

Accessing Individual trees from the random forest

In [ ]:
# Extract the first tree from the random forest to examine it individually
singleTree = rfModel.estimators_[0]
# Set the figure size to 15 inches wide by 8 inches tall
plt.figure(figsize=[15, 8])
# Create a visualization of the single decision tree with feature names and font size 10
tree.plot_tree(singleTree, feature_names=X.columns, filled=False, fontsize=10);